In [ ]:
# transformers v5 removed its TensorFlow models (TFViTModel), so stay on v4.
# tf-keras ships with Kaggle's TensorFlow image; if the version check in the next cell fails, install the
# release that matches TensorFlow, e.g. !pip install -q "tf-keras~=2.18.0" for TF 2.18.
!pip install -q "transformers>=4.40,<5"

In [ ]:
# Run this cell before anything imports TensorFlow (restart the kernel if TF was already imported).
# The HF TFViTModel is a tf-keras (Keras 2) model. Inside a Keras 3 model its weights are not tracked,
# so the ViT silently never trains. Running the whole notebook on tf-keras makes it a proper sub-layer.
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import time
SESSION_START = time.time()  # the training time budget counts from here

import random
import numpy as np
import tensorflow as tf
import transformers

import tf_keras

# tf.keras has no __version__ under tf-keras, so compare the classes instead
assert tf.keras.Model is tf_keras.Model, (
    "tf.keras is not tf-keras (Keras 2); restart the kernel and run this cell before any TensorFlow import"
)
assert int(transformers.__version__.split(".")[0]) < 5, "transformers>=5 has no TensorFlow models"
print("TensorFlow:", tf.__version__, "| tf-keras:", tf_keras.__version__, "| transformers:", transformers.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
import os
import shutil
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Apply CLAHE to improve contrast
def preprocess_image(image_path, target_size=(224, 224)):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, target_size)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(image)

# Preprocess and save images to a new directory
def preprocess_dataset(input_dir, output_dir, target_size=(224, 224)):
    os.makedirs(output_dir, exist_ok=True)
    for emotion in os.listdir(input_dir):
        emotion_dir = os.path.join(input_dir, emotion)
        output_emotion_dir = os.path.join(output_dir, emotion)
        os.makedirs(output_emotion_dir, exist_ok=True)

        for img_name in os.listdir(emotion_dir):
            img_path = os.path.join(emotion_dir, img_name)
            processed_img = preprocess_image(img_path, target_size=target_size)
            output_path = os.path.join(output_emotion_dir, img_name)
            cv2.imwrite(output_path, processed_img)

# Preprocess combined training and testing datasets
preprocessed_train = '/kaggle/temp/preprocessed/train'
preprocessed_test = '/kaggle/temp/preprocessed/test'

combined_train = '/kaggle/input/combined/combined/train'
combined_test = '/kaggle/input/combined/combined/test'

preprocess_dataset(combined_train, preprocessed_train)
preprocess_dataset(combined_test, preprocessed_test)
print("Preprocessing completed!")

In [ ]:
import hashlib

# ==============================
#   Train / Validation Split
# ==============================
# Early stopping and the LR schedule monitor this held-out slice of the training data.
# The test set is only used once, after training.

def split_train_val(src_dir, train_out, val_out, val_frac=0.1, seed=42):
    """Stratified per-class split of src_dir into train_out and val_out (files are copied)."""
    rng = np.random.default_rng(seed)
    for out_dir in (train_out, val_out):
        shutil.rmtree(out_dir, ignore_errors=True)
    for class_name in sorted(os.listdir(src_dir)):
        class_dir = os.path.join(src_dir, class_name)
        if not os.path.isdir(class_dir):
            continue
        files = sorted(os.listdir(class_dir))
        rng.shuffle(files)
        n_val = max(1, int(round(len(files) * val_frac)))
        for out_dir, subset in ((val_out, files[:n_val]), (train_out, files[n_val:])):
            out_class_dir = os.path.join(out_dir, class_name)
            os.makedirs(out_class_dir, exist_ok=True)
            for f in subset:
                shutil.copy(os.path.join(class_dir, f), out_class_dir)
        print(f" Class '{class_name}': {len(files) - n_val} train / {n_val} val")

def file_hashes(directory):
    """md5 of every file under directory (recursively)."""
    hashes = set()
    for root, _, files in os.walk(directory):
        for f in files:
            with open(os.path.join(root, f), "rb") as fh:
                hashes.add(hashlib.md5(fh.read()).hexdigest())
    return hashes

def count_md5_overlap(dir_a, dir_b, name_a="A", name_b="B"):
    """Report how many byte-identical images appear in both directories (informational only)."""
    overlap = len(file_hashes(dir_a) & file_hashes(dir_b))
    print(f"Byte-identical images shared by {name_a} and {name_b}: {overlap}")
    return overlap

preprocessed_train = '/kaggle/temp/preprocessed/train'
preprocessed_test = '/kaggle/temp/preprocessed/test'
split_train = '/kaggle/temp/split/train'
split_val = '/kaggle/temp/split/val'

split_train_val(preprocessed_train, split_train, split_val, val_frac=0.1, seed=SEED)
count_md5_overlap(split_train, split_val, "train", "val")
count_md5_overlap(split_train, preprocessed_test, "train", "test")

In [ ]:
import os
import shutil
import cv2
import numpy as np
import math
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

# ==============================
#   1) Helper Functions
# ==============================

def get_class_distribution(directory):
    """
    Dynamically discover how many images each class folder contains.
    Returns a dict: { 'class_folder_name': count_of_images, ... }
    """
    distribution = {}
    if not os.path.exists(directory):
        print(f"Directory does not exist: {directory}")
        return distribution

    classes = [d for d in os.listdir(directory) if os.path.isdir(os.path.join(directory, d))]
    for class_name in classes:
        class_dir = os.path.join(directory, class_name)
        distribution[class_name] = len(os.listdir(class_dir))
    return distribution

def oversample_based_on_distribution(
    input_dir,
    output_dir,
    class_distribution,
    target_size=(224, 224),
    fill_mode='nearest'
):
    """
    Dynamically oversample classes to match the size of the largest class.

    Each augmented copy gets a unique name (aug_<k>_<original name>.png). datagen.flow(save_to_dir=...)
    only adds a random 0-9999 suffix, so augmented images silently overwrote each other.

    Args:
        input_dir (str): Directory of the preprocessed dataset (by class folders).
        output_dir (str): Where to save the oversampled dataset (recreated on every run).
        class_distribution (dict): e.g. {'0': 4693, '1': 1153, '2': 4378, '3': 11986, ...}
        target_size (tuple): Image size (height, width).
        fill_mode (str): Fill mode for augmentation.

    Returns:
        dict: Expected number of images per class in output_dir.
    """
    datagen = ImageDataGenerator(
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        horizontal_flip=True,
        zoom_range=0.1,
        fill_mode=fill_mode
    )
    shutil.rmtree(output_dir, ignore_errors=True)
    os.makedirs(output_dir, exist_ok=True)

    # 1. Find the largest class
    max_count = max(class_distribution.values()) if class_distribution else 0
    print(f"Largest class count (pre-balancing): {max_count}")

    expected_counts = {}
    # 2. For each class, compute how many times to replicate
    for class_name, file_count in class_distribution.items():
        input_class_dir = os.path.join(input_dir, class_name)
        output_class_dir = os.path.join(output_dir, class_name)
        os.makedirs(output_class_dir, exist_ok=True)

        files = os.listdir(input_class_dir)

        # Always copy the original images
        for f in files:
            shutil.copy(os.path.join(input_class_dir, f), output_class_dir)
        expected_counts[class_name] = len(files)

        # If class has no images, skip
        if file_count == 0:
            print(f"Skipping class '{class_name}' - no images.")
            continue

        # ratio = how many times bigger the largest class is, compared to the current class
        ratio = max_count // file_count
        if ratio <= 1:
            # This class is already near or at the largest
            continue

        print(f"Class {class_name} has {file_count} images; oversampling ratio: {ratio}")

        # Generate 'ratio - 1' new images per original
        oversample_factor = ratio - 1

        for f in files:
            img = cv2.imread(os.path.join(input_class_dir, f), cv2.IMREAD_GRAYSCALE)
            if img is None:
                continue
            img = cv2.resize(img, target_size).astype('float32')[..., np.newaxis]  # (H, W, 1)
            stem = os.path.splitext(f)[0]
            for k in range(oversample_factor):
                augmented = datagen.random_transform(img)
                out_path = os.path.join(output_class_dir, f"aug_{k}_{stem}.png")
                cv2.imwrite(out_path, np.clip(augmented, 0, 255).astype('uint8'))
            expected_counts[class_name] += oversample_factor

    return expected_counts

def compute_dynamic_class_weights(generator):
    """
    Computes class weights dynamically based on the class distribution in the dataset.
    Args:
        generator (DirectoryIterator): The generator to compute class weights from.
    Returns:
        dict: Dictionary of class weights { class_index: weight, ... }
    """
    class_counts = Counter(generator.classes)
    print("Class distribution in generator:", dict(class_counts))

    classes = np.unique(generator.classes)
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=generator.classes
    )
    return dict(enumerate(class_weights))

def visualize_class_distribution_and_weights(generator, class_weights_dict, title_prefix=""):
    """
    Visualizes the class distribution and weights as plots and a table.
    Args:
        generator: The data generator used to compute class distribution.
        class_weights_dict: Dictionary of class weights.
        title_prefix: String to prepend to plot titles (e.g. "Before Balancing" / "After Balancing").
    """
    class_indices = generator.class_indices
    # Sort class names by index
    sorted_class_names = sorted(class_indices, key=class_indices.get)

    # Count the number of samples per class
    class_counts = Counter(generator.classes)

    # Build arrays for plot
    class_sizes = []
    weights = []
    for cname in sorted_class_names:
        idx = class_indices[cname]
        class_sizes.append(class_counts[idx])
        weights.append(class_weights_dict.get(idx, 1.0))

    df = pd.DataFrame({
        "Class Name": sorted_class_names,
        "Number of Images": class_sizes,
        "Class Weight": weights
    })

    # Display the table
    print(f"\n{title_prefix} Class Distribution and Weights:")
    print(df)

    # Plot the class sizes
    plt.figure(figsize=(12, 5))
    plt.bar(sorted_class_names, class_sizes, color='skyblue')
    plt.title(f"{title_prefix} Class Sizes")
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=45)
    plt.show()

    # Plot the class weights
    plt.figure(figsize=(12, 5))
    plt.bar(sorted_class_names, weights, color='orange')
    plt.title(f"{title_prefix} Class Weights")
    plt.xlabel("Class")
    plt.ylabel("Weight")
    plt.xticks(rotation=45)
    plt.show()


# Paths (only the training split is oversampled; validation and test stay untouched)
split_train = '/kaggle/temp/split/train'
oversampled_train = '/kaggle/temp/oversampled/train'

# 1. Compute & Print Distribution BEFORE Balancing
pre_balancing_dist = get_class_distribution(split_train)
print("\nClass Distribution (Before Balancing):")
for k, v in pre_balancing_dist.items():
    print(f" Class '{k}': {v} images")

# 2. Oversample
expected_counts = oversample_based_on_distribution(
    input_dir=split_train,
    output_dir=oversampled_train,
    class_distribution=pre_balancing_dist,  # Computed distribution
    target_size=(224, 224),
    fill_mode='nearest'
)

# 3. Compute & Print Distribution AFTER Balancing
post_balancing_dist = get_class_distribution(oversampled_train)
print("\nClass Distribution (After Balancing):")
for k, v in post_balancing_dist.items():
    print(f" Class '{k}': {v} images")
assert post_balancing_dist == expected_counts, (
    f"Oversampled counts {post_balancing_dist} differ from expected {expected_counts}; files were overwritten"
)

# 4. Create Generators & Compute Weights
img_size = (224, 224)
batch_size = 16

train_datagen = ImageDataGenerator(rescale=1.0/255.0)
oversampled_generator = train_datagen.flow_from_directory(
    oversampled_train,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=True
)

class_weights_dict = compute_dynamic_class_weights(oversampled_generator)
print("\nComputed Class Weights (After Balancing):")
print(class_weights_dict)

# 5. Visualize final distribution & weights
visualize_class_distribution_and_weights(oversampled_generator, class_weights_dict, title_prefix="After Balancing")


In [ ]:
import os
import shutil
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import pandas as pd

import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    GlobalAveragePooling2D, Dense, Dropout,
    BatchNormalization, Conv2D, Multiply,
    Lambda, Concatenate, Reshape, Input
)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve
)
from sklearn.preprocessing import label_binarize

# ---- NEW IMPORTS FOR ViT ----
from transformers import TFViTModel

In [ ]:

class ViTWrapper(tf.keras.layers.Layer):
    """
    A custom Keras layer that wraps TFViTModel and ensures compatibility with the functional API.
    """
    def __init__(self, pretrained_name="google/vit-base-patch16-224-in21k", **kwargs):
        super().__init__(**kwargs)
        self.pretrained_name = pretrained_name
        self.vit_model = TFViTModel.from_pretrained(pretrained_name)

    def call(self, inputs, training=False):
        # Transpose to channels-first for ViT
        x = tf.transpose(inputs, perm=[0, 3, 1, 2])  # (B, 3, 224, 224)
        # Normalize inputs: the in21k checkpoint was trained with mean = std = 0.5
        x = (x - 0.5) / 0.5
        outputs = self.vit_model(pixel_values=x, training=training)
        return outputs.pooler_output

    def freeze(self):
        """Freeze the whole ViT. The model-level flag overrides every sub-layer."""
        self.vit_model.trainable = False

    def unfreeze_layers(self):
        """Unfreeze the whole ViT."""
        self.vit_model.trainable = True

    def get_config(self):
        config = super().get_config()
        config.update({"pretrained_name": self.pretrained_name})
        return config


In [ ]:
# -----------------------------
# Squash Activation Function for Capsules
# -----------------------------
def squash(s, axis=-1, epsilon=1e-7):
    s_squared_norm = tf.reduce_sum(tf.square(s), axis=axis, keepdims=True)
    scale = s_squared_norm / (1 + s_squared_norm)
    return scale * s / tf.sqrt(s_squared_norm + epsilon)

# -----------------------------
# Advanced Capsule Fusion Block (CFB)
# -----------------------------
class AdvancedCapsuleFusionBlock(tf.keras.layers.Layer):
    """
    Advanced Capsule Fusion Block that fuses two feature vectors using an iterative dynamic routing
    algorithm with enhanced transformations, residual updates, dropout, and batch normalization.
    """
    def __init__(self, capsule_dim=256, num_iterations=2, dropout_rate=0.4, **kwargs):
        super(AdvancedCapsuleFusionBlock, self).__init__(**kwargs)
        self.capsule_dim = capsule_dim
        self.num_iterations = num_iterations
        self.dropout_rate = dropout_rate
        
        # Two-layer MLP for each branch's transformation
        self.mlp1 = tf.keras.Sequential([
            Dense(capsule_dim, activation="relu"),
            Dropout(dropout_rate),
            Dense(capsule_dim, activation=None)
        ])
        self.bn1 = BatchNormalization()
        
        self.mlp2 = tf.keras.Sequential([
            Dense(capsule_dim, activation="relu"),
            Dropout(dropout_rate),
            Dense(capsule_dim, activation=None)
        ])
        self.bn2 = BatchNormalization()
    
    def call(self, inputs, training=False):
        # inputs: list [f1, f2] from two branches
        f1, f2 = inputs
        # Transform each branch using a two-layer MLP
        u1 = self.bn1(self.mlp1(f1), training=training)  # (batch, capsule_dim)
        u2 = self.bn2(self.mlp2(f2), training=training)  # (batch, capsule_dim)
        
        # Stack to form capsules: shape (batch, 2, capsule_dim)
        u = tf.stack([u1, u2], axis=1)
        
        # Initialize routing logits b (batch, 2) with zeros
        b = tf.zeros(shape=(tf.shape(u)[0], 2), dtype=tf.float32)
        
        # Iterative dynamic routing
        for i in range(self.num_iterations):
            c = tf.nn.softmax(b, axis=1)  # coupling coefficients (batch, 2)
            c_expanded = tf.expand_dims(c, axis=-1)  # (batch, 2, 1)
            # Weighted sum of capsule predictions
            s = tf.reduce_sum(c_expanded * u, axis=1)  # (batch, capsule_dim)
            v = squash(s, axis=-1)  # (batch, capsule_dim)
            
            # Residual update: encourage consistency with previous capsule output
            if i > 0:
                v = 0.5 * v + 0.5 * v_prev
            v_prev = v  # store for residual update
            
            v_expanded = tf.expand_dims(v, axis=1)  # (batch, 1, capsule_dim)
            # Update routing logits based on agreement
            b += tf.reduce_sum(u * v_expanded, axis=-1)  # (batch, 2)
        return v  # Final fused capsule (batch, capsule_dim)


In [ ]:
# -----------------------------
# Positional Embedding Layer for TRB
# -----------------------------
class PositionalEmbedding(tf.keras.layers.Layer):
    """
    Learns a positional embedding for a fixed number of tokens.
    """
    def __init__(self, num_tokens, token_dim, **kwargs):
        super(PositionalEmbedding, self).__init__(**kwargs)
        self.num_tokens = num_tokens
        self.token_dim = token_dim

    def build(self, input_shape):
        # Create a learnable positional embedding matrix
        self.pos_embedding = self.add_weight(
            shape=(1, self.num_tokens, self.token_dim),
            initializer="uniform",
            trainable=True,
            name="pos_embedding"
        )
        super(PositionalEmbedding, self).build(input_shape)

    def call(self, inputs):
        # inputs shape: (batch, num_tokens, token_dim)
        return inputs + self.pos_embedding

# -----------------------------
# Advanced Transformer Encoder Block for TRB
# -----------------------------
class TransformerEncoderBlock(tf.keras.layers.Layer):
    """
    A full transformer encoder block with multi-head self-attention, feed-forward network,
    residual connections, and layer normalization.
    """
    def __init__(self, token_dim, num_heads, ff_dim, dropout_rate=0.4, **kwargs):
        super(TransformerEncoderBlock, self).__init__(**kwargs)
        self.mha = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=token_dim)
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(token_dim)
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)
    
    def call(self, inputs, training=False):
        attn_output = self.mha(inputs, inputs, training=training)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# -----------------------------
# Advanced Transformer Refinement Block (TRB)
# -----------------------------
class AdvancedTransformerRefinementBlock(tf.keras.layers.Layer):
    """
    Refines the fused capsule features using a stack of transformer encoder blocks.
    The fused vector is projected into multiple tokens, positional embeddings are added,
    processed by transformer layers, and then aggregated.
    """
    def __init__(self, num_tokens=4, token_dim=64, num_heads=4, ff_dim=128, num_layers=2, dropout_rate=0.3, **kwargs):
        super(AdvancedTransformerRefinementBlock, self).__init__(**kwargs)
        self.num_tokens = num_tokens
        self.token_dim = token_dim
        self.proj = Dense(num_tokens * token_dim)
        self.reshape_layer = Lambda(lambda x: tf.reshape(x, (-1, num_tokens, token_dim)))
        self.pos_emb = PositionalEmbedding(num_tokens, token_dim)
        self.transformer_layers = [TransformerEncoderBlock(token_dim, num_heads, ff_dim, dropout_rate) for _ in range(num_layers)]
    
    def call(self, inputs, training=False):
        # inputs: fused capsule features (batch, capsule_dim)
        tokens = self.proj(inputs)                # (batch, num_tokens * token_dim)
        tokens = self.reshape_layer(tokens)         # (batch, num_tokens, token_dim)
        tokens = self.pos_emb(tokens)               # Add positional embedding
        for layer in self.transformer_layers:
            tokens = layer(tokens, training=training)
        # Aggregate tokens via mean pooling
        refined = tf.reduce_mean(tokens, axis=1)    # (batch, token_dim)
        return refined

# focal loss

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.utils import Sequence

class MixupSequence(Sequence):
    def __init__(self, datagen, alpha=0.1, **kwargs):
        """
        Wraps a Keras ImageDataGenerator (Sequence) to perform MixUp on each batch.

        Args:
            datagen: A Keras ImageDataGenerator Sequence (e.g., from flow_from_directory).
            alpha: Beta distribution parameter for mixing. Higher alpha means stronger mixing.
        """
        super().__init__(**kwargs)
        self.datagen = datagen
        self.alpha = alpha
        self.batch_size = datagen.batch_size
        self.steps = len(datagen)

    def __len__(self):
        # Return the number of batches per epoch
        return self.steps

    def __getitem__(self, idx):
        # Get the idx-th batch from the underlying generator
        batch_x, batch_y = self.datagen[idx]
        batch_size = batch_x.shape[0]

        # Sample lambda from Beta(alpha, alpha)
        lam = np.random.beta(self.alpha, self.alpha)

        # Shuffle batch indices
        index = np.random.permutation(batch_size)

        # Perform MixUp on images and labels
        mixed_x = lam * batch_x + (1 - lam) * batch_x[index, :]
        mixed_y = lam * batch_y + (1 - lam) * batch_y[index, :]

        return mixed_x, mixed_y

    def on_epoch_end(self):
        # Let the wrapped generator reshuffle; otherwise every epoch repeats the first epoch's order
        self.datagen.on_epoch_end()


In [ ]:
def focal_loss(alpha, gamma=2.0):
    """
    Focal Loss for multi-class classification.
    
    Args:
        alpha: A list or array of weights for each class (class importance).
        gamma: Focusing parameter to reduce the loss contribution from easy examples.
    
    Returns:
        A loss function that can be used in model.compile().
    """
    def loss_fn(y_true, y_pred):
        # Ensure predictions are probabilities
        y_pred = tf.keras.backend.clip(y_pred, 1e-3, 1 - 1e-3)
        
        # Compute the cross-entropy loss
        ce_loss = -y_true * tf.math.log(y_pred)
        
        # Apply the focal loss adjustment
        focal_loss = alpha * tf.pow(1 - y_pred, gamma) * ce_loss
        
        # Return the mean focal loss over the batch
        return tf.reduce_mean(tf.reduce_sum(focal_loss, axis=-1))
    
    return loss_fn


In [ ]:


# =================================
# 1) Paths and Data Generators
# =================================

oversampled_train = '/kaggle/temp/oversampled/train'
split_val = '/kaggle/temp/split/val'
# The test set is used untouched: no oversampling, no augmentation, and only after training
preprocessed_test = '/kaggle/temp/preprocessed/test'

img_size = (224, 224)
batch_size = 16

# Train Data Augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=15,       # Increased rotation range
    width_shift_range=0.1,   # Increased shift
    height_shift_range=0.1,
    brightness_range=[0.8, 1.0],  # Vary brightness
    shear_range=0.1,              # Shear transformations
    zoom_range=0.15,               # Increased zoom range
    horizontal_flip=True,
    fill_mode='reflect'
)

# Validation/Test: rescale only
test_datagen = ImageDataGenerator(rescale=1.0/255.0)

# Create Generators
train_generator = train_datagen.flow_from_directory(
    oversampled_train,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=True,
    seed=SEED
)
val_generator = test_datagen.flow_from_directory(
    split_val,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)
test_generator = test_datagen.flow_from_directory(
    preprocessed_test,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)
assert train_generator.class_indices == val_generator.class_indices == test_generator.class_indices

# =================================
# 2) Class Weight Computation
# =================================

def compute_dynamic_class_weights(generator):
    class_counts = Counter(generator.classes)
    print("Class distribution in generator:", dict(class_counts))
    classes = np.unique(generator.classes)
    class_weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=generator.classes
    )
    return dict(enumerate(class_weights))

def visualize_class_distribution_and_weights(generator, class_weights_dict, title_prefix=""):
    class_indices = generator.class_indices
    sorted_class_names = sorted(class_indices, key=class_indices.get)

    # Gather data
    class_counts = Counter(generator.classes)
    class_sizes = []
    weights = []
    for cname in sorted_class_names:
        idx = class_indices[cname]
        class_sizes.append(class_counts[idx])
        weights.append(class_weights_dict.get(idx, 1.0))

    # Create a DataFrame for printing
    df = pd.DataFrame({
        "Class Name": sorted_class_names,
        "Number of Images": class_sizes,
        "Class Weight": weights
    })
    print(f"\n{title_prefix} Class Distribution and Weights:")
    print(df)

    # Plot Class Sizes
    plt.figure(figsize=(12, 5))
    plt.bar(sorted_class_names, class_sizes, color='skyblue')
    plt.title(f"{title_prefix} Class Sizes")
    plt.xlabel("Class")
    plt.ylabel("Number of Images")
    plt.xticks(rotation=45)
    plt.show()

    # Plot Class Weights
    plt.figure(figsize=(12, 5))
    plt.bar(sorted_class_names, weights, color='orange')
    plt.title(f"{title_prefix} Class Weights")
    plt.xlabel("Class")
    plt.ylabel("Weight")
    plt.xticks(rotation=45)
    plt.show()

class_weights_dict = compute_dynamic_class_weights(train_generator)
visualize_class_distribution_and_weights(train_generator, class_weights_dict, title_prefix="Oversampled Train")

# =================================
# 3) CBAM + VGG16 + ViT (Multi-Branch) Model Definition
# =================================

def attention_block(input_tensor):
    # Channel Attention
    avg_pool = GlobalAveragePooling2D()(input_tensor)  # (batch, channels)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=[1, 2], keepdims=False))(input_tensor)  # (batch, channels)

    reduced_channels = input_tensor.shape[-1] // 8  # Reduce dimensionality
    avg_dense = Dense(reduced_channels, activation="relu")(avg_pool)
    max_dense = Dense(reduced_channels, activation="relu")(max_pool)
    channel_attention = Dense(input_tensor.shape[-1], activation="sigmoid")(avg_dense + max_dense)
    channel_refined = Multiply()([input_tensor, Reshape((1, 1, -1))(channel_attention)])

    # Spatial Attention
    avg_pool = Lambda(lambda x: tf.reduce_mean(x, axis=-1, keepdims=True))(channel_refined)
    max_pool = Lambda(lambda x: tf.reduce_max(x, axis=-1, keepdims=True))(channel_refined)
    concat = Concatenate(axis=-1)([avg_pool, max_pool])
    spatial_attention = Conv2D(1, (7, 7), activation="sigmoid", padding="same")(concat)
    refined_output = Multiply()([channel_refined, spatial_attention])
    return refined_output


class VGGPreprocess(tf.keras.layers.Layer):
    """
    Converts the [0, 1] RGB model input into the caffe preprocessing VGG16's ImageNet weights
    were trained with: 0-255 range, BGR channel order, ImageNet mean subtracted.
    """
    def call(self, inputs):
        x = inputs * 255.0
        x = x[..., ::-1]  # RGB -> BGR
        return x - tf.constant([103.939, 116.779, 123.68], dtype=x.dtype)


def build_vgg16_vit_model(num_classes):
    """
    Creates a multi-branch model that fuses features from:
      - VGG16 + CBAM CNN branch
      - Vision Transformer (ViT) branch
    Fusion is performed using an advanced Capsule Fusion Block followed by an advanced Transformer Refinement Block.
    """
    input_tensor = Input(shape=(224, 224, 3), name="image_input")

    # VGG16 + CBAM Branch
    base_model = VGG16(weights="imagenet", include_top=False, input_shape=(224, 224, 3))
    for layer in base_model.layers[:-8]:
        layer.trainable = False
    vgg_input = VGGPreprocess(name="vgg_preprocess")(input_tensor)
    vgg_features = base_model(vgg_input)
    vgg_features = attention_block(vgg_features)
    vgg_features = GlobalAveragePooling2D()(vgg_features)
    vgg_features = Dense(256, activation="relu")(vgg_features)

    # ViT Branch (normalizes its own input)
    vit_branch = ViTWrapper(pretrained_name="google/vit-base-patch16-224-in21k", name="vit_wrapper")
    vit_features = vit_branch(input_tensor)

    # Fuse features using the Advanced Capsule Fusion Block
    fused_capsule = AdvancedCapsuleFusionBlock(capsule_dim=256, num_iterations=2, dropout_rate=0.4)([vgg_features, vit_features])

    # Refine fused features with the Advanced Transformer Refinement Block
    refined_features = AdvancedTransformerRefinementBlock(num_tokens=4, token_dim=64, num_heads=4, ff_dim=128, num_layers=2, dropout_rate=0.4)(fused_capsule)

    # Final Classification Layer
    final_output = Dense(num_classes, activation="softmax")(refined_features)

    model = Model(inputs=input_tensor, outputs=final_output, name="Advanced_VGG16_CBAM_ViT_Model")
    return model


def count_vit_trainable_weights(model):
    """Number of ViT weight tensors the optimizer will update (0 means the ViT is frozen)."""
    n_vit = len(model.get_layer("vit_wrapper").trainable_weights)
    print(f"Trainable weight tensors: ViT {n_vit}, whole model {len(model.trainable_weights)}")
    return n_vit

# =================================
# 4) Training Utilities
# =================================

def evaluate_model(model, test_generator, history):
    # Evaluate on test data
    test_loss, test_accuracy = model.evaluate(test_generator)
    print(f"\nFinal Test Loss: {test_loss:.4f}")
    print(f"Final Test Accuracy: {test_accuracy:.4f}")

    # Accuracy & Loss Plots
    plt.figure(figsize=(12, 5))

    # Accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history['accuracy'], label='Train Accuracy')
    plt.plot(history['val_accuracy'], label='Validation Accuracy')
    plt.title('Model Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    # Loss
    plt.subplot(1, 2, 2)
    plt.plot(history['loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Model Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.show()

    # Predict once; the probabilities feed the report, confusion matrix and ROC
    test_generator.reset()
    y_pred_prob = model.predict(test_generator)
    y_pred = np.argmax(y_pred_prob, axis=1)
    y_true = test_generator.classes
    class_labels = list(test_generator.class_indices.keys())

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_labels))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels)
    plt.title("Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

    # ROC AUC
    y_true_binarized = label_binarize(y_true, classes=range(len(class_labels)))
    roc_auc = roc_auc_score(y_true_binarized, y_pred_prob, average="macro", multi_class="ovr")
    print(f"Macro-average ROC AUC: {roc_auc:.4f}")

    # Plot ROC Curve
    plt.figure(figsize=(10, 7))
    for i in range(len(class_labels)):
        fpr, tpr, _ = roc_curve(y_true_binarized[:, i], y_pred_prob[:, i])
        per_class_auc = roc_auc_score(y_true_binarized[:, i], y_pred_prob[:, i])
        plt.plot(fpr, tpr, label=f"{class_labels[i]} (AUC = {per_class_auc:.2f})")
    plt.plot([0, 1], [0, 1], 'k--', label='Random Chance')
    plt.title('ROC Curve')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend()
    plt.grid()
    plt.show()

# Normalize class weights for Focal Loss
alpha = np.array(list(class_weights_dict.values()))
alpha /= np.sum(alpha)

import numpy as np
import tensorflow as tf
from sklearn.metrics import accuracy_score, classification_report
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Define the modular loss function factory
def get_optimized_loss(alpha_weights, gamma_value=2.0):
    """
    Returns a compiled Focal Loss function.
    Setting gamma >= 2.0 aggressively suppresses gradients from easy majority classes (like Happy/Neutral).
    """
    return focal_loss(alpha=alpha_weights, gamma=gamma_value)

# 2. TTA Function: accepts an in-memory model or a saved-weights path
def evaluate_with_tta(model, test_generator, tta_steps=5):
    print("\n==========================================")
    print("Starting Test-Time Augmentation (TTA)...")
    print("==========================================")

    # If a weights path is passed instead of a model object, rebuild the model and load the weights
    if isinstance(model, str):
        print(f"Loading weights from disk: {model}")
        weights_path = model
        model = build_vgg16_vit_model(len(test_generator.class_indices))
        model.load_weights(weights_path).expect_partial()

    # Reset generator to start from the first batch
    test_generator.reset()
    y_true = test_generator.classes
    num_samples = len(y_true)
    num_classes = len(test_generator.class_indices)

    # Accumulate probability predictions across passes
    tta_predictions = np.zeros((num_samples, num_classes))

    # Step 1: Base unaugmented predictions
    print(f"Pass 1/{tta_steps}: Running base prediction pass...")
    test_generator.reset()
    tta_predictions += model.predict(test_generator, verbose=1)

    # Step 2: Configure light TTA transformations (flips and tiny zooms)
    tta_datagen = ImageDataGenerator(
        rescale=1.0/255.0,
        horizontal_flip=True,
        zoom_range=0.08
    )

    tta_generator = tta_datagen.flow_from_directory(
        test_generator.directory,
        target_size=test_generator.target_size,
        batch_size=test_generator.batch_size,
        class_mode="categorical",
        shuffle=False
    )

    # Step 3: Run augmented passes
    for step in range(1, tta_steps):
        print(f"Pass {step + 1}/{tta_steps}: Running augmented prediction pass...")
        tta_generator.reset()
        tta_predictions += model.predict(tta_generator, verbose=1)

    # Average the softmax probabilities across all passes
    tta_predictions /= tta_steps
    final_preds = np.argmax(tta_predictions, axis=1)

    acc = accuracy_score(y_true, final_preds)
    print("\n---------------------------------------------------------")
    print(f" FINAL TTA BOOSTED TEST ACCURACY: {acc:.4f} ")
    print("---------------------------------------------------------")
    print(classification_report(y_true, final_preds, target_names=list(test_generator.class_indices.keys())))
    return acc


## Training across Kaggle sessions (12-hour limit)
Every epoch saves the full training state to `/kaggle/working/checkpoints` (this version's output). A session stops
starting new epochs after `TIME_BUDGET_HOURS`, so Kaggle can save that output before the 12-hour limit.

To continue in a new session:
1. Open this notebook in the editor and add the finished version's output as an input (**Add Input**, then pick
   this notebook's output, or a Dataset made from it).
2. **Save Version → Save & Run All** again. The training cell finds `checkpoints/latest.json` under `/kaggle/input`
   and resumes at the next epoch, with the same optimizer state, learning rate, early-stopping counters and history.

Evaluation, TTA and the cross-dataset test run in the session that finishes training. Intermediate images are
written to `/kaggle/temp`, so each version's output is only the checkpoints.

In [ ]:
# =================================
# Checkpointing across Kaggle sessions
# =================================
import glob
import json

CHECKPOINT_DIR = "/kaggle/working/checkpoints"
RESUME = True               # set False to ignore previous checkpoints and train from scratch
RESUME_SEARCH = ["/kaggle/input/**/checkpoints/latest.json", CHECKPOINT_DIR + "/latest.json"]
TIME_BUDGET_HOURS = 11.0    # no new epoch starts after this; leaves time for evaluation and saving the output
KEEP_LATEST = 2             # full checkpoints kept (older ones are deleted to save disk space)
HISTORY_KEYS = ("accuracy", "val_accuracy", "loss", "val_loss")

def seconds_left():
    """Seconds left in this session's training budget."""
    return TIME_BUDGET_HOURS * 3600 - (time.time() - SESSION_START)

def layer_checkpoint(model, **extra):
    """
    tf.train.Checkpoint over the model's layers. Weights are matched by object structure, not by weight order
    (the order changes when layers are frozen or unfrozen). The optimizer is only included when passed in.
    """
    return tf.train.Checkpoint(**{f"layer_{i:03d}": layer for i, layer in enumerate(model.layers)}, **extra)

def save_weights_only(model, name):
    layer_checkpoint(model).write(os.path.join(CHECKPOINT_DIR, name))

def load_weights_only(model, name):
    layer_checkpoint(model).read(os.path.join(CHECKPOINT_DIR, name)).expect_partial()

def save_full_state(model, name):
    layer_checkpoint(model, optimizer=model.optimizer).write(os.path.join(CHECKPOINT_DIR, name))

def load_full_state(model, name):
    """Weights plus optimizer state (Adam moments, step count, learning rate), to continue the same phase."""
    model.optimizer.build(model.trainable_variables)  # create the optimizer slots so they can be filled
    status = layer_checkpoint(model, optimizer=model.optimizer).read(os.path.join(CHECKPOINT_DIR, name))
    status.assert_existing_objects_matched()

def find_resume_state():
    """The most advanced training state saved by a previous session, or None."""
    states = []
    for pattern in RESUME_SEARCH:
        for path in glob.glob(pattern, recursive=True):
            with open(path) as fh:
                state = json.load(fh)
            state["source_dir"] = os.path.dirname(path)
            states.append(state)
    if not states:
        return None
    return max(states, key=lambda s: (s["phase"], s["phase_done"], s["next_epoch"]))

def import_checkpoint_files(state):
    """Copy the previous session's checkpoint files into this session's output, so the next session finds them."""
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    if os.path.abspath(state["source_dir"]) != os.path.abspath(CHECKPOINT_DIR):
        for f in os.listdir(state["source_dir"]):
            shutil.copy(os.path.join(state["source_dir"], f), CHECKPOINT_DIR)

def prune_checkpoints():
    """Keep only the KEEP_LATEST newest full checkpoints (names sort by phase, then epoch)."""
    prefixes = sorted({f.split(".")[0] for f in os.listdir(CHECKPOINT_DIR) if f.startswith("ckpt_phase")})
    for prefix in prefixes[:-KEEP_LATEST]:
        for f in glob.glob(os.path.join(CHECKPOINT_DIR, prefix + ".*")):
            os.remove(f)


class ResumableEarlyStopping(EarlyStopping):
    """EarlyStopping that can continue from a saved patience counter, best score and best weights."""
    def __init__(self, resume_state=None, **kwargs):
        super().__init__(**kwargs)
        self.resume_state = resume_state

    def on_train_begin(self, logs=None):
        super().on_train_begin(logs)  # resets the counters
        if self.resume_state:
            self.wait = self.resume_state["wait"]
            self.best = self.resume_state["best"]
            self.best_weights = self.resume_state["best_weights"]


class ResumableReduceLROnPlateau(ReduceLROnPlateau):
    """ReduceLROnPlateau that can continue from a saved patience counter, best score and cooldown."""
    def __init__(self, resume_state=None, **kwargs):
        super().__init__(**kwargs)
        self.resume_state = resume_state

    def on_train_begin(self, logs=None):
        super().on_train_begin(logs)  # resets the counters
        if self.resume_state:
            self.wait = self.resume_state["wait"]
            self.best = self.resume_state["best"]
            self.cooldown_counter = self.resume_state["cooldown_counter"]


class SessionCheckpoint(tf.keras.callbacks.Callback):
    """
    After every epoch, saves:
      - the full state (weights + optimizer) as ckpt_phase<P>_epoch<E>
      - the phase's best weights as phase<P>_best
      - latest.json with the epoch counter, callback states and history
    It stops the phase early if the next epoch would not finish inside the session's time budget.
    Put it last in the callback list so it sees EarlyStopping's and ReduceLROnPlateau's updates.
    """
    def __init__(self, phase_config, early_stopping, reduce_lr, state):
        super().__init__()
        self.phase_config = phase_config
        self.early_stopping = early_stopping
        self.reduce_lr = reduce_lr
        self.state = state  # shared training state, updated in place
        self.paused = False
        self.epoch_start = None

    def on_epoch_begin(self, epoch, logs=None):
        self.epoch_start = time.time()

    def on_epoch_end(self, epoch, logs=None):
        phase = self.phase_config["phase"]
        state = self.state
        for key in HISTORY_KEYS:
            state["history"][key].append(float(logs[key]))
        if logs["val_loss"] < state["phase_best"]:
            state["phase_best"] = float(logs["val_loss"])
            save_weights_only(self.model, f"phase{phase}_best")

        # EarlyStopping runs earlier in the list: it sets stop_training and restores the best weights
        phase_done = bool(self.model.stop_training) or epoch + 1 >= self.phase_config["epochs"]
        name = f"ckpt_phase{phase}_epoch{epoch + 1:02d}"
        save_full_state(self.model, name)
        state.update({
            "phase": phase,
            "next_epoch": epoch + 1,
            "phase_done": phase_done,
            "checkpoint": name,
            "early_stopping": {"wait": int(self.early_stopping.wait), "best": float(self.early_stopping.best)},
            "reduce_lr": {"wait": int(self.reduce_lr.wait), "best": float(self.reduce_lr.best),
                          "cooldown_counter": int(self.reduce_lr.cooldown_counter)},
            "learning_rate": float(tf.keras.backend.get_value(self.model.optimizer.learning_rate)),
            "epoch_seconds": time.time() - self.epoch_start,
            "saved_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        })
        if phase_done:
            state["phase_epochs"][str(phase)] = epoch + 1
        tmp_path = os.path.join(CHECKPOINT_DIR, "latest.json.tmp")
        with open(tmp_path, "w") as fh:
            json.dump({k: v for k, v in state.items() if k != "source_dir"}, fh, indent=1)
        os.replace(tmp_path, os.path.join(CHECKPOINT_DIR, "latest.json"))
        prune_checkpoints()

        if not phase_done and seconds_left() < 1.2 * state["epoch_seconds"]:
            self.paused = True
            self.model.stop_training = True
            print(f"\nSession time budget reached: stopped after phase {phase}, epoch {epoch + 1}. "
                  "Attach this version's output and run again to continue.")

In [ ]:
# =================================
# 5) Two-phase training that resumes across sessions
# =================================

# Parallel batch loading: ImageDataGenerator augmentation runs on the CPU
FIT_KWARGS = dict(workers=4, max_queue_size=20)

PHASES = [
    # Phase 1: ViT frozen; VGG16 blocks 4-5, CBAM, fusion blocks and head train
    {"phase": 1, "learning_rate": 3e-5, "epochs": 25, "vit_trainable": False, "unfreeze_all": False},
    # Phase 2: unfreeze everything, including VGG16 blocks 1-3 and the ViT
    {"phase": 2, "learning_rate": 1e-5, "epochs": 35, "vit_trainable": True, "unfreeze_all": True},
]
loss_fn = get_optimized_loss(alpha_weights=alpha, gamma_value=2.0)

def configure_phase(model, phase_config):
    """Set the phase's trainable layers and compile with a fresh AdamW, as in the original two-phase schedule."""
    if phase_config["unfreeze_all"]:
        for layer in model.layers:
            layer.trainable = True
    vit = model.get_layer("vit_wrapper")
    if phase_config["vit_trainable"]:
        vit.unfreeze_layers()
    else:
        vit.freeze()
    n_vit = count_vit_trainable_weights(model)
    assert (n_vit > 0) == phase_config["vit_trainable"], "ViT trainable state is wrong; check the tf.keras version cell"
    model.compile(
        optimizer=AdamW(learning_rate=phase_config["learning_rate"], weight_decay=1e-3),
        loss=loss_fn,
        metrics=["accuracy"]
    )

def train_model():
    """
    Runs, or continues, the two-phase schedule.
    Returns (model, state) when training is complete, or (None, state) if this session's time budget ran out first.
    """
    model = build_vgg16_vit_model(train_generator.num_classes)
    mixup_train_sequence = MixupSequence(train_generator, alpha=0.1)
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)

    state = find_resume_state() if RESUME else None
    if state is None:
        print("No previous checkpoint found: training from scratch.")
        state = {"phase": 1, "next_epoch": 0, "phase_done": False, "phase_epochs": {},
                 "history": {key: [] for key in HISTORY_KEYS}}
        weights_loaded = True  # fresh model, nothing to load
    else:
        print(f"Resuming from {state['source_dir']}: phase {state['phase']}, {state['next_epoch']} epochs done"
              f"{' (phase finished)' if state['phase_done'] else ''}, saved {state['saved_at']}")
        import_checkpoint_files(state)
        weights_loaded = False

    for phase_config in PHASES:
        phase = phase_config["phase"]
        if phase < state["phase"] or (phase == state["phase"] and state["phase_done"]):
            continue  # finished in an earlier session
        if seconds_left() < 2 * state.get("epoch_seconds", 0):
            print(f"Not enough session time left to start phase {phase}; attach this version's output and run again.")
            return None, state

        configure_phase(model, phase_config)
        early_stopping = ResumableEarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True)
        reduce_lr = ResumableReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6)
        if phase == state["phase"] and state["next_epoch"] > 0:
            # Continue the phase the last session stopped in: the phase's best weights go to EarlyStopping,
            # then the latest weights + optimizer state, epoch counter and callback states are restored
            load_weights_only(model, f"phase{phase}_best")
            early_stopping.resume_state = dict(state["early_stopping"], best_weights=model.get_weights())
            reduce_lr.resume_state = state["reduce_lr"]
            load_full_state(model, state["checkpoint"])
            initial_epoch = state["next_epoch"]
        else:
            if not weights_loaded:
                load_weights_only(model, state["checkpoint"])  # weights at the end of the previous phase
            state["phase_best"] = float("inf")
            initial_epoch = 0
        weights_loaded = True

        saver = SessionCheckpoint(phase_config, early_stopping, reduce_lr, state)
        print(f"\n=== Phase {phase}: epochs {initial_epoch + 1}-{phase_config['epochs']} ===")
        model.fit(
            mixup_train_sequence,
            validation_data=val_generator,
            initial_epoch=initial_epoch,
            epochs=phase_config["epochs"],
            callbacks=[early_stopping, reduce_lr, saver],
            **FIT_KWARGS
        )
        if saver.paused:
            return None, state

    if not weights_loaded:
        # Training finished in an earlier session: load the final weights for evaluation
        configure_phase(model, PHASES[-1])
        load_weights_only(model, state["checkpoint"])
    return model, state

# =================================
# 6) Run Training & Evaluation
# =================================

if __name__ == "__main__":
    final_model, training_state = train_model()
    if final_model is None:
        print("\nTraining is not finished. Evaluation, TTA and the cross-dataset test run in the session "
              "that finishes training.")
    else:
        # Standard Evaluation (first use of the test set)
        evaluate_model(final_model, test_generator, training_state["history"])

        # Save final weights (TF checkpoint format; reload with build_vgg16_vit_model + load_weights)
        save_path = "vgg16_cbam_vit_incremental_unfreeze_improved"
        final_model.save_weights(save_path)
        print(f"\nWeights saved successfully to {save_path}")

        # Test-Time Augmentation on the in-memory model
        evaluate_with_tta(final_model, test_generator, tta_steps=5)

# cross validate

In [ ]:
import os
import shutil
import cv2
import numpy as np
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

# Apply CLAHE to improve contrast
def preprocess_image(image_path, target_size=(224, 224)):
    image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    image = cv2.resize(image, target_size)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    return clahe.apply(image)

# Preprocess and save images to a new directory
def preprocess_dataset(input_dir, output_dir, target_size=(224, 224)):
    os.makedirs(output_dir, exist_ok=True)
    for emotion in os.listdir(input_dir):
        emotion_dir = os.path.join(input_dir, emotion)
        output_emotion_dir = os.path.join(output_dir, emotion)
        os.makedirs(output_emotion_dir, exist_ok=True)

        for img_name in os.listdir(emotion_dir):
            img_path = os.path.join(emotion_dir, img_name)
            processed_img = preprocess_image(img_path, target_size=target_size)
            output_path = os.path.join(output_emotion_dir, img_name)
            cv2.imwrite(output_path, processed_img)

# Preprocess testing datasets
cross_test = '/kaggle/temp/cross/test'

combined_test = '/kaggle/input/emotion-detection-fer/test'

preprocess_dataset(combined_test, cross_test)
print("Preprocessing completed!")

In [ ]:
# Cross-Validation on a New Dataset (FERPlus as an example)
new_dataset_path =  '/kaggle/temp/cross/test'

# Create Test Generator
new_test_generator = test_datagen.flow_from_directory(
    new_dataset_path,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="categorical",
    shuffle=False
)

# Folders are matched to the training classes by sorted position ('disgust' vs 'disgusted', etc.)
train_class_names = sorted(train_generator.class_indices, key=train_generator.class_indices.get)
new_class_names = sorted(new_test_generator.class_indices, key=new_test_generator.class_indices.get)
assert len(train_class_names) == len(new_class_names), (train_class_names, new_class_names)
for train_name, new_name in zip(train_class_names, new_class_names):
    assert new_name.startswith(train_name[:4]), f"Class mismatch: {train_name} vs {new_name}"

# If the combined training data already contains these images, this is not a true cross-dataset test
count_md5_overlap('/kaggle/temp/preprocessed/train', new_dataset_path, "combined train", "emotion-detection-fer test")

# Evaluate on the New Dataset
def evaluate_on_new_dataset(model, test_generator, dataset_name="New Dataset"):
    """
    Evaluate the trained model on another dataset.
    """
    test_loss, test_accuracy = model.evaluate(test_generator)
    print(f"\n{dataset_name} - Test Loss: {test_loss:.4f}")
    print(f"{dataset_name} - Test Accuracy: {test_accuracy:.4f}")

    # Classification Report
    y_pred = np.argmax(model.predict(test_generator), axis=1)
    y_true = test_generator.classes
    class_labels = list(test_generator.class_indices.keys())

    print(f"\n{dataset_name} - Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_labels))

    # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_labels, yticklabels=class_labels)
    plt.title(f"{dataset_name} - Confusion Matrix")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.show()

# Evaluate on FERPlus (or any other dataset), once training has finished
if final_model is not None:
    evaluate_on_new_dataset(final_model, new_test_generator, dataset_name="emotion-detection-fer")
else:
    print("Training is not finished in this session; the cross-dataset test runs after the final session.")
